In [1]:
import pyspark
from pyspark import SparkContext
from pyspark.sql import Row, SQLContext
from pyspark.sql.functions import count, col
from pyspark import SparkFiles
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, VectorIndexer
from pyspark.ml.linalg import Vectors
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, GBTClassifier, OneVsRest
from pyspark.ml.evaluation import MulticlassClassificationEvaluator,  BinaryClassificationEvaluator
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

L'objectif de cet exercice est de lire et de convertir les données du dataset Iris en un DataFrame Spark, afin d'entraîner et de tester différents classificateurs sur ces données.
L'utilisation de Spark permet de charger et de transformer les données rapidement grâce à un pipeline de transformation, utilisant StringIndexer et VectorAssembler, qui permettent de traiter les données en parallèle.

In [2]:
# Initialisation de la saison spark
spark = SparkSession.builder.appName("IrisClassification").getOrCreate()

25/03/02 14:24:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
# Création du dataframe Spark

## On définit un schema pour indiquer le type de chaque feature
schema = StructType([
    StructField("sepal_length", DoubleType()),
    StructField("sepal_width", DoubleType()),
    StructField("petal_length", DoubleType()),
    StructField("petal_width", DoubleType()),
    StructField("species", StringType())
])


pandas_data = pd.read_csv("iris.csv")

# Convertir en DataFrame Spark avec schéma pre-defini
data = spark.createDataFrame(pandas_data, schema=schema)

data.printSchema()
data.show(5)

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- species: string (nullable = true)

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| Setosa|
|         4.9|        3.0|         1.4|        0.2| Setosa|
|         4.7|        3.2|         1.3|        0.2| Setosa|
|         4.6|        3.1|         1.5|        0.2| Setosa|
|         5.0|        3.6|         1.4|        0.2| Setosa|
+------------+-----------+------------+-----------+-------+
only showing top 5 rows



In [4]:
# Description statistique du dataframe
data.describe().show()

+-------+------------------+------------------+------------------+------------------+---------+
|summary|      sepal_length|       sepal_width|      petal_length|       petal_width|  species|
+-------+------------------+------------------+------------------+------------------+---------+
|  count|               150|               150|               150|               150|      150|
|   mean| 5.843333333333334|3.0573333333333323|3.7579999999999996|1.1993333333333336|     null|
| stddev|0.8280661279778636|0.4358662849366982| 1.765298233259466|0.7622376689603465|     null|
|    min|               4.3|               2.0|               1.0|               0.1|   Setosa|
|    max|               7.9|               4.4|               6.9|               2.5|Virginica|
+-------+------------------+------------------+------------------+------------------+---------+



In [6]:
# Repartion des especes dans le dataset
total_count = data.count()

species_counts = data.groupBy("species").agg(
    count("species").alias("count")
).withColumn("percentage", (col("count") / total_count) * 100)
species_counts.show()

+----------+-----+-----------------+
|   species|count|       percentage|
+----------+-----+-----------------+
| Virginica|   50|33.33333333333333|
|    Setosa|   50|33.33333333333333|
|Versicolor|   50|33.33333333333333|
+----------+-----+-----------------+



Le dataset est équilibré

In [7]:
# Encodage de la target column
stringIndexer = StringIndexer(inputCol="species", outputCol="numeric_species")

# Rassemblement des diffents descripteurs dans un seul vecteur pour chaque sample du dataframe
vectorAssembler = VectorAssembler(inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"],\
                                  outputCol="features")

In [8]:
## split data
(training_data, test_data) = data.randomSplit([0.8, 0.2])

In [9]:
# Entraînement DecisonTree

## Initialisation du modèle
dt_cls = DecisionTreeClassifier(labelCol="numeric_species", featuresCol="features")

## pre-processing et model dans une pipeline
pipeline = Pipeline(stages=[stringIndexer, vectorAssembler, dt_cls])

## entraîement du modèle
model = pipeline.fit(training_data)

## inference
dt_predictions = model.transform(test_data)
dt_predictions.select("prediction", "numeric_species", "features").show()

+----------+---------------+-----------------+
|prediction|numeric_species|         features|
+----------+---------------+-----------------+
|       0.0|            0.0|[5.1,3.5,1.4,0.2]|
|       0.0|            0.0|[5.0,3.4,1.5,0.2]|
|       0.0|            0.0|[5.4,3.4,1.5,0.4]|
|       0.0|            0.0|[4.4,3.0,1.3,0.2]|
|       0.0|            0.0|[4.9,3.6,1.4,0.1]|
|       0.0|            0.0|[5.0,3.5,1.3,0.3]|
|       2.0|            2.0|[6.9,3.1,4.9,1.5]|
|       2.0|            2.0|[5.2,2.7,3.9,1.4]|
|       2.0|            2.0|[5.9,3.0,4.2,1.5]|
|       2.0|            2.0|[6.0,2.2,4.0,1.0]|
|       2.0|            2.0|[5.6,3.0,4.5,1.5]|
|       2.0|            2.0|[6.1,2.8,4.0,1.3]|
|       2.0|            2.0|[6.2,2.2,4.5,1.5]|
|       2.0|            2.0|[5.7,2.6,3.5,1.0]|
|       1.0|            2.0|[6.7,3.0,5.0,1.7]|
|       2.0|            2.0|[6.0,3.4,4.5,1.6]|
|       2.0|            2.0|[6.7,3.1,4.7,1.5]|
|       2.0|            2.0|[5.7,2.9,4.2,1.3]|
|       2.0| 

In [10]:
# Evaluation

dt_evaluator = MulticlassClassificationEvaluator(
    labelCol="numeric_species", predictionCol="prediction", metricName="accuracy")

dt_accuracy = dt_evaluator.evaluate(dt_predictions)
print("Test accuracy = ", dt_accuracy*100, "%")

Test accuracy =  93.75 %


In [11]:
# Entraînement Random Forest

rf = RandomForestClassifier(labelCol="numeric_species", featuresCol="features")

rf_pipeline = Pipeline(stages=[stringIndexer, vectorAssembler, rf])

rf_model = rf_pipeline.fit(training_data)

rf_predictions = rf_model.transform(test_data)

rf_predictions.select("prediction", "numeric_species", "features").show()

rf_evaluator = MulticlassClassificationEvaluator(
    labelCol="numeric_species", predictionCol="prediction", metricName="accuracy")

rf_accuracy = rf_evaluator.evaluate(rf_predictions)
print("Test accuracy = ", rf_accuracy*100, "%")

+----------+---------------+-----------------+
|prediction|numeric_species|         features|
+----------+---------------+-----------------+
|       0.0|            0.0|[5.1,3.5,1.4,0.2]|
|       0.0|            0.0|[5.0,3.4,1.5,0.2]|
|       0.0|            0.0|[5.4,3.4,1.5,0.4]|
|       0.0|            0.0|[4.4,3.0,1.3,0.2]|
|       0.0|            0.0|[4.9,3.6,1.4,0.1]|
|       0.0|            0.0|[5.0,3.5,1.3,0.3]|
|       2.0|            2.0|[6.9,3.1,4.9,1.5]|
|       2.0|            2.0|[5.2,2.7,3.9,1.4]|
|       2.0|            2.0|[5.9,3.0,4.2,1.5]|
|       2.0|            2.0|[6.0,2.2,4.0,1.0]|
|       2.0|            2.0|[5.6,3.0,4.5,1.5]|
|       2.0|            2.0|[6.1,2.8,4.0,1.3]|
|       2.0|            2.0|[6.2,2.2,4.5,1.5]|
|       2.0|            2.0|[5.7,2.6,3.5,1.0]|
|       1.0|            2.0|[6.7,3.0,5.0,1.7]|
|       2.0|            2.0|[6.0,3.4,4.5,1.6]|
|       2.0|            2.0|[6.7,3.1,4.7,1.5]|
|       2.0|            2.0|[5.7,2.9,4.2,1.3]|
|       2.0| 

In [13]:
# Gradient Boosting

## transformer le classificateur binaire en une multi-classe classifieur à l’aide d’un classificateur d’arbres

gbt = GBTClassifier(labelCol="numeric_species", featuresCol="features")

ovr = OneVsRest(classifier=gbt, labelCol="numeric_species")
gbt_ovr_pipeline = Pipeline(stages=[stringIndexer, vectorAssembler, ovr])

gbt_ovr_model = gbt_ovr_pipeline.fit(training_data)

gbt_ovr_predictions = gbt_ovr_model.transform(test_data)

gbt_ovr_predictions.select("prediction", "numeric_species", "features").show()

gbt_ovr_evaluator = MulticlassClassificationEvaluator(
    labelCol="numeric_species", predictionCol="prediction", metricName="accuracy")

gbt_over_accuracy = gbt_ovr_evaluator.evaluate(gbt_ovr_predictions)
print("Test accuracy = ", gbt_over_accuracy*100, "%")



+----------+---------------+-----------------+
|prediction|numeric_species|         features|
+----------+---------------+-----------------+
|       0.0|            0.0|[5.1,3.5,1.4,0.2]|
|       0.0|            0.0|[5.0,3.4,1.5,0.2]|
|       0.0|            0.0|[5.4,3.4,1.5,0.4]|
|       0.0|            0.0|[4.4,3.0,1.3,0.2]|
|       0.0|            0.0|[4.9,3.6,1.4,0.1]|
|       0.0|            0.0|[5.0,3.5,1.3,0.3]|
|       2.0|            2.0|[6.9,3.1,4.9,1.5]|
|       2.0|            2.0|[5.2,2.7,3.9,1.4]|
|       2.0|            2.0|[5.9,3.0,4.2,1.5]|
|       2.0|            2.0|[6.0,2.2,4.0,1.0]|
|       2.0|            2.0|[5.6,3.0,4.5,1.5]|
|       2.0|            2.0|[6.1,2.8,4.0,1.3]|
|       2.0|            2.0|[6.2,2.2,4.5,1.5]|
|       2.0|            2.0|[5.7,2.6,3.5,1.0]|
|       1.0|            2.0|[6.7,3.0,5.0,1.7]|
|       2.0|            2.0|[6.0,3.4,4.5,1.6]|
|       2.0|            2.0|[6.7,3.1,4.7,1.5]|
|       2.0|            2.0|[5.7,2.9,4.2,1.3]|
|       2.0| 

25/03/02 14:27:54 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 175410 ms exceeds timeout 120000 ms
25/03/02 14:27:54 WARN SparkContext: Killing executors is not supported by current scheduler.


In [25]:
spark.stop()